## Objetivo:
Treinar modelos Prophet para prever o **consumo diário** de cada item.

A partir dessas previsões de consumo diário, o serviço calculará:
1. **Estoque futuro** - Deduzindo o consumo previsto do estoque atual

2. **Necessidade de reposição** - Quando o estoque atingir o mínimo**Abordagem**: Cada item terá seu próprio modelo Prophet treinado com dados de consumo diário.

3. **Quantidade a repor** - Para manter níveis adequados

In [ ]:
# %pip install -q pandas numpy prophet matplotlib seaborn psycopg2-binary sqlalchemy
# %pip install -q prophet

import pandas as pd
import numpy as np
import os
import sys
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from prophet import Prophet
from prophet.plot import plot_plotly, plot_components_plotly
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
from pathlib import Path
import pickle

warnings.filterwarnings('ignore')

try:
    # O notebook está em 'ai/notebooks', então o diretório 'ai' é o pai.
    # É o diretório 'ai' que precisa estar no path para que 'from src...' funcione.
    here = Path().resolve()
    ai_dir = here.parent 
    
    if str(ai_dir) not in sys.path:
        sys.path.insert(0, str(ai_dir))
        print(f"✅ Diretório '{ai_dir.name}' adicionado ao sys.path.")
        
except Exception as e:
    print(f"❌ Erro ao configurar o sys.path: {e}")

print("✅ Ambiente configurado com sucesso!")


: 

# 2. Carregamento dos Dados de Consumo Diário

Vamos carregar e agregar os dados de consumo por dia e item.

In [ ]:
try:
    from src.utils.database import DatabaseConnector
    print("✅ Módulo 'DatabaseConnector' importado com sucesso.")
    
    # Conexão com o banco
    db = DatabaseConnector()
    
    # Carregar dados brutos
    raw_data = db.load_raw_data()
    
    if raw_data.empty:
        raise ValueError("Nenhum dado retornado. Verifique a conexão e a query.")
    
    print(f"✅ {len(raw_data)} registros brutos carregados.")
    
    # Agregar consumo diário por item
    daily_consumption = raw_data.groupby(['item_id', 'order_date']).agg({
        'quantity': 'sum',
        'current_stock': 'last',
        'minimum_stock': 'first',
        'maximum_stock': 'first',
        'item_name': 'first'
    }).reset_index()
    
    daily_consumption.columns = ['item_id', 'ds', 'daily_quantity', 'current_stock', 'minimum_stock', 'maximum_stock', 'item_name']
    daily_consumption = daily_consumption.sort_values(['item_id', 'ds']).reset_index(drop=True)
    
    print(f"\n✅ {len(daily_consumption)} registros de consumo diário criados.")
    print(f"📦 Itens únicos: {daily_consumption['item_id'].nunique()}")
    print(f"📅 Período: {daily_consumption['ds'].min()} até {daily_consumption['ds'].max()}")
    
    print("\nPrimeiras linhas dos dados de consumo diário:")
    display(daily_consumption.head(10))
    
    print("\n" + "="*60)
    print("ESTATÍSTICAS DESCRITIVAS")
    print("="*60)
    display(daily_consumption[['daily_quantity', 'current_stock']].describe())
    
    print("\n" + "="*60)
    print("ANÁLISE DE VALORES NULOS")
    print("="*60)
    null_counts = daily_consumption.isnull().sum()
    null_columns = null_counts[null_counts > 0]
    
    if len(null_columns) > 0:
        print(f"⚠️  Encontradas {len(null_columns)} colunas com valores nulos:")
        print(null_columns)
    else:
        print("✅ Nenhum valor nulo encontrado no dataset!")

except ImportError as e:
    print(f"❌ Falha ao importar 'DatabaseConnector': {e}")
except Exception as e:
    print(f"❌ Erro ao carregar ou analisar os dados: {e}")

# 3. Análise Exploratória dos Dados (EDA)

Vamos entender os padrões nos dados brutos.

In [ ]:
if 'daily_consumption' in locals() and not daily_consumption.empty:
    sns.set_style("whitegrid")
    
    # Agrupar por data para visualização geral
    ts_data = daily_consumption.groupby('ds').agg({
        'daily_quantity': 'sum',
        'current_stock': 'sum'
    }).reset_index()

    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    
    # 1. Consumo diário total
    ax1 = axes[0, 0]
    ts_data.plot(x='ds', y='daily_quantity', ax=ax1, color='skyblue', legend=None)
    ax1.set_title('Consumo Diário Total (Todos os Itens)', fontweight='bold')
    ax1.set_xlabel('Data')
    ax1.set_ylabel('Quantidade Consumida por Dia')
    
    # 2. Média móvel de 7 dias do consumo
    ax2 = axes[0, 1]
    ts_data['ma7'] = ts_data['daily_quantity'].rolling(window=7, min_periods=1).mean()
    ts_data.plot(x='ds', y='ma7', ax=ax2, color='orange', legend=None)
    ax2.set_title('Média Móvel (7 dias) do Consumo', fontweight='bold')
    ax2.set_xlabel('Data')
    ax2.set_ylabel('Consumo Médio')

    # 3. Distribuição de consumo diário
    ax3 = axes[1, 0]
    daily_consumption['daily_quantity'].hist(bins=50, ax=ax3, color='lightgreen', edgecolor='black')
    ax3.set_title('Distribuição do Consumo Diário', fontweight='bold')
    ax3.set_xlabel('Quantidade Diária')
    ax3.set_ylabel('Frequência')

    # 4. Top 10 itens por consumo total
    ax4 = axes[1, 1]
    top_items = daily_consumption.groupby('item_id')['daily_quantity'].sum().nlargest(10)
    top_items.plot(kind='barh', ax=ax4, color='coral')
    ax4.set_title('Top 10 Itens por Consumo Total', fontweight='bold')
    ax4.set_xlabel('Quantidade Total')
    ax4.set_ylabel('Item ID')
    ax4.invert_yaxis()
    
    plt.tight_layout()
    plt.show()
    
    # Análise de padrões semanais
    daily_consumption['day_of_week'] = pd.to_datetime(daily_consumption['ds']).dt.dayofweek
    weekly_pattern = daily_consumption.groupby('day_of_week')['daily_quantity'].mean()
    
    fig, ax = plt.subplots(figsize=(10, 6))
    weekly_pattern.plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title('Padrão de Consumo por Dia da Semana', fontweight='bold')
    ax.set_xlabel('Dia da Semana (0=Segunda, 6=Domingo)')
    ax.set_ylabel('Consumo Médio')
    ax.set_xticklabels(['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sáb', 'Dom'], rotation=0)
    plt.tight_layout()
    plt.show()
    
else:
    print("❌ Dados não carregados. Execute a célula anterior.")

# 4. Engenharia de Features

Vamos criar features agregadas por item e período para o treinamento.

In [ ]:
# Esta célula não é mais necessária, pois a engenharia de features foi movida para `database.py`
# e a preparação para o Prophet é feita na célula de treinamento.
pass


# 5. Preparação dos Dados para Treinamento

Dividir em treino/teste e normalizar.

In [ ]:
# Esta célula não é mais necessária. A divisão treino/teste é feita dentro
# do loop de treinamento do Prophet por item.
pass


# 6. Treinamento dos Modelos

Vamos testar 3 algoritmos diferentes e escolher o melhor.

In [ ]:
# Funções: treinar Prophet por item para prever consumo diário

def train_prophet_daily_consumption(item_df: pd.DataFrame, forecast_days: int = 30, min_days: int = 14):
    """
    Treina modelo Prophet para prever consumo diário de um item.
    
    Args:
        item_df: DataFrame com colunas ['ds', 'daily_quantity', 'item_id']
        forecast_days: Número de dias para prever (padrão: 30)
        min_days: Mínimo de dias de histórico necessários (padrão: 14)
    
    Returns:
        tuple: (modelo treinado, DataFrame com previsões) ou (None, None) se falhar
    """
    item_id = int(item_df['item_id'].iloc[0])
    
    # Reindexar para garantir frequência diária e preencher buracos
    item_df = item_df.set_index('ds').asfreq('D')
    item_df['item_id'] = item_id
    item_df['daily_quantity'] = item_df['daily_quantity'].fillna(0)  # Dias sem pedido = consumo 0
    item_df = item_df.reset_index()
    
    if len(item_df) < min_days:
        return None, None
    
    # Preparar para Prophet
    df_prophet = item_df[['ds', 'daily_quantity']].rename(columns={'daily_quantity': 'y'})
    
    # Criar features adicionais
    df_prophet['day_of_week'] = df_prophet['ds'].dt.dayofweek
    df_prophet['is_weekend'] = (df_prophet['day_of_week'] >= 5).astype(int)
    
    # Médias móveis para capturar tendência
    df_prophet['ma_7'] = df_prophet['y'].rolling(window=7, min_periods=1).mean()
    df_prophet['ma_14'] = df_prophet['y'].rolling(window=14, min_periods=1).mean()
    
    # Configurar Prophet
    m = Prophet(
        daily_seasonality=True,
        weekly_seasonality=True,
        yearly_seasonality='auto',
        changepoint_prior_scale=0.05,  # Menos sensível a mudanças abruptas
        seasonality_prior_scale=10.0
    )
    
    # Adicionar regressores
    m.add_regressor('is_weekend')
    m.add_regressor('ma_7')
    
    try:
        m.fit(df_prophet.dropna())
    except Exception as e:
        print(f"⚠️ Erro ao treinar item {item_id}: {e}")
        return None, None
    
    # Fazer previsão
    future = m.make_future_dataframe(periods=forecast_days, freq='D')
    
    # Preencher regressores para o futuro
    future['day_of_week'] = future['ds'].dt.dayofweek
    future['is_weekend'] = (future['day_of_week'] >= 5).astype(int)
    
    # Para ma_7, usar últimos valores conhecidos
    last_ma7 = df_prophet['ma_7'].iloc[-1]
    future['ma_7'] = future['ds'].apply(lambda x: last_ma7 if x > df_prophet['ds'].max() else df_prophet.loc[df_prophet['ds'] == x, 'ma_7'].values[0] if x in df_prophet['ds'].values else last_ma7)
    
    forecast = m.predict(future)
    forecast['yhat'] = forecast['yhat'].clip(lower=0)  # Consumo não pode ser negativo
    
    return m, forecast

# ===== TREINAMENTO EM LOTE =====

print("="*80)
print("🚀 INICIANDO TREINAMENTO DE MODELOS")
print("="*80)

if 'daily_consumption' not in locals() or daily_consumption.empty:
    print("❌ Execute a célula de carregamento de dados primeiro!")
else:
    # Configurações
    n_items = 200  # Número de itens para treinar (ajuste conforme necessário)
    forecast_days = 30  # Prever 30 dias à frente
    min_days = 14  # Mínimo de 14 dias de histórico
    
    # Selecionar itens com mais dados
    item_counts = daily_consumption.groupby('item_id').size().sort_values(ascending=False)
    items_to_train = item_counts[item_counts >= min_days].head(n_items).index.tolist()
    
    print(f"📊 Itens selecionados para treinamento: {len(items_to_train)}")
    print(f"📅 Previsão: {forecast_days} dias à frente")
    print(f"📈 Mínimo de dias históricos: {min_days}\n")
    
    trained_models = {}
    training_errors = []
    
    for i, item_id in enumerate(items_to_train, 1):
        if i % 20 == 0:
            print(f"Progresso: {i}/{len(items_to_train)} itens processados...")
        
        df_item = daily_consumption[daily_consumption['item_id'] == item_id].copy()
        
        model, forecast = train_prophet_daily_consumption(df_item, forecast_days, min_days)
        
        if model is not None:
            trained_models[item_id] = {
                'model': model,
                'forecast': forecast,
                'training_days': len(df_item),
                'item_name': df_item['item_name'].iloc[0] if 'item_name' in df_item.columns else f"Item {item_id}"
            }
        else:
            training_errors.append(item_id)
    
    print("\n" + "="*80)
    print("📊 RESULTADOS DO TREINAMENTO")
    print("="*80)
    print(f"✅ Modelos treinados com sucesso: {len(trained_models)}")
    print(f"❌ Falhas no treinamento: {len(training_errors)}")
    
    if trained_models:
        # Estatísticas de treinamento
        training_days = [v['training_days'] for v in trained_models.values()]
        print(f"\n📈 Dias de histórico usado:")
        print(f"   Média: {np.mean(training_days):.1f} dias")
        print(f"   Mínimo: {np.min(training_days)} dias")
        print(f"   Máximo: {np.max(training_days)} dias")
        
        # Exemplo de previsão
        sample_item = list(trained_models.keys())[0]
        sample_forecast = trained_models[sample_item]['forecast']
        future_forecast = sample_forecast[sample_forecast['ds'] > daily_consumption['ds'].max()].head(7)
        
        print(f"\n🔮 Exemplo de previsão (Item {sample_item}, próximos 7 dias):")
        display(future_forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].round(2))

# 7. Análise dos Resultados

Visualizar a performance dos modelos.

In [ ]:
# 7. VISUALIZAÇÃO DOS RESULTADOS

if 'trained_models' in locals() and trained_models:
    # Escolher um item aleatório que foi treinado para visualizar
    item_id_to_plot = np.random.choice(list(trained_models.keys()))
    model_data = trained_models[item_id_to_plot]
    model_to_plot = model_data['model']
    forecast = model_data['forecast']
    item_name = model_data['item_name']
    
    print(f"📊 Visualizando previsões para: {item_name} (ID: {item_id_to_plot})")
    print(f"📈 Dias de histórico: {model_data['training_days']}")

    # Gráfico de previsão
    fig1 = plot_plotly(model_to_plot, forecast)
    fig1.update_layout(
        title=f'Previsão de Consumo Diário - {item_name}',
        xaxis_title='Data',
        yaxis_title='Consumo Diário (unidades)'
    )
    fig1.show()

    # Gráfico de componentes (tendência, sazonalidade, etc.)
    fig2 = plot_components_plotly(model_to_plot, forecast)
    fig2.update_layout(title=f'Componentes da Previsão - {item_name}')
    fig2.show()
    
    # Análise de erros no histórico (in-sample)
    df_cv = forecast.set_index('ds')[['yhat']].join(model_to_plot.history.set_index('ds')[['y']])
    df_cv = df_cv.dropna()
    
    r2 = r2_score(df_cv['y'], df_cv['yhat'])
    mae = mean_absolute_error(df_cv['y'], df_cv['yhat'])
    rmse = np.sqrt(mean_squared_error(df_cv['y'], df_cv['yhat']))
    
    print("\n" + "="*80)
    print(f"📊 MÉTRICAS DE AJUSTE (IN-SAMPLE)")
    print("="*80)
    print(f"   Item: {item_name} (ID: {item_id_to_plot})")
    print(f"   R² Score: {r2:.4f}")
    print(f"   MAE (Erro Médio Absoluto): {mae:.2f} unidades/dia")
    print(f"   RMSE (Raiz do Erro Quadrático): {rmse:.2f} unidades/dia")
    
    # Mostrar resumo das previsões futuras
    future_only = forecast[forecast['ds'] > model_to_plot.history['ds'].max()].copy()
    
    print(f"\n🔮 RESUMO DAS PREVISÕES (próximos {len(future_only)} dias):")
    print(f"   Consumo médio previsto: {future_only['yhat'].mean():.2f} unidades/dia")
    print(f"   Consumo total previsto: {future_only['yhat'].sum():.2f} unidades")
    print(f"   Pico de consumo: {future_only['yhat'].max():.2f} unidades/dia")
    print(f"   Menor consumo: {future_only['yhat'].min():.2f} unidades/dia")
    
else:
    print("❌ Nenhum modelo treinado para visualizar. Execute a célula de treinamento.")

# 8. Salvar o Modelo Treinado

In [ ]:
# 8. SALVAR OS MODELOS TREINADOS

if 'trained_models' in locals() and trained_models:
    print("="*80)
    print("💾 SALVANDO MODELOS PROPHET")
    print("="*80)
    
    try:
        model_dir = Path('../models')
        model_dir.mkdir(exist_ok=True)
        
        # Salvar cada modelo treinado (apenas o modelo, não o forecast completo)
        for item_id, data in trained_models.items():
            model_path = model_dir / f'prophet_daily_consumption_{item_id}.pkl'
            with open(model_path, 'wb') as f:
                pickle.dump(data['model'], f)
        
        print(f"✅ {len(trained_models)} modelos salvos no diretório: {model_dir}")
        
        # Salvar metadados gerais
        metadata = {
            'model_type': 'prophet_daily_consumption',
            'prediction_target': 'daily_consumption',
            'training_date': pd.Timestamp.now().isoformat(),
            'num_models': len(trained_models),
            'forecast_days': forecast_days,
            'min_days_required': min_days,
            'items_trained': list(trained_models.keys()),
            'item_info': {
                item_id: {
                    'name': data['item_name'],
                    'training_days': data['training_days']
                }
                for item_id, data in trained_models.items()
            }
        }
        
        metadata_path = model_dir / 'prophet_daily_metadata.pkl'
        with open(metadata_path, 'wb') as f:
            pickle.dump(metadata, f)
        print(f"✅ Metadados salvos: {metadata_path.name}")
        print(f"\n📋 Informações salvas:")
        print(f"   - Tipo de modelo: Previsão de Consumo Diário")
        print(f"   - Número de modelos: {len(trained_models)}")
        print(f"   - Horizonte de previsão: {forecast_days} dias")

    except Exception as e:
        print(f"❌ Erro ao salvar: {e}")
else:
    print("❌ Nenhum modelo treinado para salvar.")

## Reformulação: Previsão de Estoques com Prophet (Meta)

Este notebook foi reformulado para usar exclusivamente o Prophet (Meta) para prever estoques futuros por item.

Regras principais:
- O target será `current_stock` (snapshot mensal) — queremos prever estoques futuros.
- Cada item precisa de pelo menos 3 meses de registro para gerar uma previsão; itens com menos dados serão ignorados.
- Usamos agregados mensais retornados por `DatabaseConnector.load_monthly_features()`.
- Resultado: previsões por item para um horizonte configurável (p.ex. 6 meses) e opção de salvar no banco.


In [ ]:
# Setup e carregamento das features mensais via DatabaseConnector

# Instala/garante Prophet (descomente se precisar):
# %pip install -q prophet

import pandas as pd
import numpy as np
from prophet import Prophet
from datetime import datetime

# carregar DatabaseConnector atualizado
try:
    from ai.src.utils.database import DatabaseConnector
    db = DatabaseConnector()
    print('✅ DatabaseConnector inicializado')
except Exception as e:
    db = None
    print('⚠️ DatabaseConnector não disponível:', e)

# Carregar features mensais agregadas (pré-processadas no banco ou calculadas)
if db is None:
    raise RuntimeError('DatabaseConnector não está disponível. Configure a conexão antes de prosseguir.')

monthly_features = db.load_monthly_features()
if monthly_features.empty:
    raise RuntimeError('monthly_features está vazio — verifique os dados no banco ou o intervalo de datas.')

print(f"✅ monthly_features carregado: {len(monthly_features)} registros, {monthly_features['item_id'].nunique()} items únicos")

# Converter year_month (YYYYMM) em índice datetime (first day do mês)
monthly_features['year_month'] = monthly_features['year_month'].astype(int)
monthly_features['ds'] = pd.to_datetime(monthly_features['year_month'].astype(str) + '01', format='%Y%m%d', errors='coerce')

# Checar colunas mínimas
required_cols = ['item_id','year_month','ds','current_stock','total_quantity']
for c in required_cols:
    if c not in monthly_features.columns:
        raise RuntimeError(f"Coluna obrigatória faltando em monthly_features: {c}")

# ordenar
monthly_features = monthly_features.sort_values(['item_id','ds']).reset_index(drop=True)

# quick preview
display(monthly_features.head())


In [ ]:
# Funções: treinar Prophet por item e salvar previsões

from sklearn.metrics import mean_absolute_error, mean_squared_error


def run_prophet_for_item(item_df: pd.DataFrame, horizon: int = 6, require_months: int = 3):
    """Recebe df mensal do item (index ds) com colunas current_stock e total_quantity.
    Retorna lista de dicts [{'item_id':..., 'target_month':'YYYY-MM', 'stock_predicted':float}, ...]
    """
    item_id = int(item_df['item_id'].iloc[0])
    if item_df.dropna(subset=['current_stock']).shape[0] < require_months:
        # não há dados suficientes
        return []

    # preparar df para Prophet
    dfp = item_df[['ds','current_stock','total_quantity']].rename(columns={'current_stock':'y','total_quantity':'demand'})
    dfp = dfp.set_index('ds').asfreq('MS')
    dfp = dfp.reset_index()

    # criar regressors simples (lags) com shift para evitar data leakage
    dfp['demand_lag_1'] = dfp['demand'].shift(1).fillna(0)
    dfp['demand_ma_3'] = dfp['demand'].shift(1).rolling(3, min_periods=1).mean().fillna(0)

    # dropna para treino
    train_df = dfp.dropna(subset=['y'])
    if train_df.shape[0] < require_months:
        return []

    m = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
    # adicionar regressors
    m.add_regressor('demand_lag_1')
    m.add_regressor('demand_ma_3')

    try:
        m.fit(train_df.rename(columns={'ds':'ds','y':'y'}))
    except Exception as e:
        print(f"Erro ajustando Prophet para item {item_id}: {e}")
        return []

    future = m.make_future_dataframe(periods=horizon, freq='MS')
    # preencher regressors futuros com última observação disponível (est. simples)
    last_vals = train_df[['demand_lag_1','demand_ma_3']].iloc[-1].to_dict()
    for reg in ['demand_lag_1','demand_ma_3']:
        future[reg] = future['ds'].apply(lambda _: float(last_vals.get(reg,0)))

    fcst = m.predict(future)
    preds = fcst[['ds','yhat']].tail(horizon).copy()
    preds['yhat'] = preds['yhat'].clip(lower=0)

    out = []
    for _, row in preds.iterrows():
        out.append({'item_id': item_id, 'target_month': row['ds'].strftime('%Y-%m'), 'stock_predicted': float(row['yhat']), 'model_used':'prophet'})
    return out


# Executar Prophet para os primeiros N items (configurável)
n_items = 200  # ajuste conforme recursos
horizon = 6
require_months = 3

items = monthly_features['item_id'].unique()[:n_items]
all_predictions = []

for it in items:
    df_item = monthly_features[monthly_features['item_id']==it].copy()
    if df_item.empty:
        continue
    # garantir índice ds
    df_item = df_item.set_index('ds')
    preds = run_prophet_for_item(df_item.reset_index(), horizon=horizon, require_months=require_months)
    if preds:
        all_predictions.extend(preds)

print(f"✅ Previsões geradas: {len(all_predictions)} (itens x meses)")

# salvar no banco se possível
if db is not None and all_predictions:
    try:
        db.save_predictions(all_predictions)
        print('✅ Previsões salvas no banco.')
    except Exception as e:
        print('❌ Falha ao salvar previsões:', e)
else:
    print('ℹ️ Banco não configurado ou sem previsões — verifique `DatabaseConnector` se desejar salvar.')

# mostrar amostra de previsões
pd.DataFrame(all_predictions).head(20)


# 9. Backtesting (Walk-Forward Validation)

Para avaliar a performance real do modelo, usamos a validação walk-forward. Este método simula como o modelo se comportaria em produção:
1. Treinamos o modelo com dados até um certo ponto no tempo (ex: 24 meses).
2. Fazemos uma previsão para os próximos `h` meses (ex: 6 meses).
3. Comparamos a previsão com os valores reais.
4. "Andamos" para frente no tempo (ex: 6 meses) e repetimos o processo, incluindo os dados que acabamos de "ver" no novo conjunto de treino.

Isso nos dá uma estimativa realista do erro esperado para cada passo do horizonte de previsão.


In [ ]:
from prophet.diagnostics import cross_validation, performance_metrics

# Escolher um item para o backtesting detalhado
# Idealmente, um item com um bom histórico de dados
item_counts = monthly_features['item_id'].value_counts()
item_to_backtest = item_counts.index[0] if not item_counts.empty else None

if item_to_backtest:
    print(f"Executando backtesting para o item: {item_to_backtest}")
    
    df_item_test = monthly_features[monthly_features['item_id'] == item_to_backtest].copy()
    
    # Preparar dataframe para Prophet
    df_prophet_test = df_item_test[['ds', 'current_stock']].rename(columns={'current_stock': 'y'})
    
    # Adicionar regressoras se houver
    df_prophet_test['demand_lag_1'] = df_item_test['total_quantity'].shift(1).fillna(0)
    df_prophet_test['demand_ma_3'] = df_item_test['total_quantity'].shift(1).rolling(3, min_periods=1).mean().fillna(0)
    df_prophet_test = df_prophet_test.dropna()

    # Instanciar o modelo
    m_test = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
    m_test.add_regressor('demand_lag_1')
    m_test.add_regressor('demand_ma_3')
    
    # Ajustar o modelo aos dados completos para usar na validação cruzada
    m_test.fit(df_prophet_test)

    # Executar a validação cruzada (cross_validation)
    # initial: período inicial de treino (ex: 24 meses)
    # period: a cada quanto tempo o corte de treino é incrementado (ex: 6 meses)
    # horizon: quantos passos à frente prever (ex: 6 meses)
    initial_train_period = '730 days' # 2 anos
    period_increment = '180 days' # 6 meses
    forecast_horizon = '180 days' # 6 meses

    df_cv = cross_validation(m_test, initial=initial_train_period, period=period_increment, horizon=forecast_horizon)

    print("\nPrimeiras linhas do resultado da validação cruzada:")
    display(df_cv.head())

    # Calcular métricas de performance
    df_p = performance_metrics(df_cv)
    
    print("\nMétricas de performance por horizonte:")
    display(df_p)

    # Visualizar o erro (MAPE) ao longo do horizonte
    from prophet.plot import plot_cross_validation_metric
    fig = plot_cross_validation_metric(df_cv, metric='mape')
    fig.update_layout(title=f'Erro Percentual Absoluto Médio (MAPE) - Item {item_to_backtest}')
    fig.show()

else:
    print("❌ Nenhum item com dados suficientes para o backtesting.")
